# Часть 1. Загрузка и предобработка данных

Перед началом хотелось бы определить направление нашей работы и вопросы на которые мы хотим ответить.

Цель данного анализа - провести исчерпывающий анализ данных о фильмах и сериалах стриммингового сервиса Netflix, на основе которого можно принять "правильное" бизнес-решение в сфере искусства и развлечений. А именно, ответить на следующий вопрос: "В какой тип контента стоит вложить свои средства? Как определить "успешный" проект?".


В этом ноутбуке мы загружаем исходный датасет `NetflixShows`, изучаем его структуру, устраняем проблемы качества данных, такие как дубликаты, пропуски и избыточные признаки.

Результат этой части - это очищенный датасет `netflix_clean_1.csv`, который используется в последующих частях анализа.

#### Выполнил Ефимов Алексей


## Шаг 1. Загрузка и первичный осмотр данных


In [77]:
import pandas as pd

data = pd.read_csv("NetflixShows.csv", encoding="utf-8-sig", sep=";")
data.head(3)

,title,rating,ratingLevel,ratingDescription,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",80,2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",100,2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,90,2016,98.0,80


Представленный датасет содержит следующие признаки (колонки):

- `title` - название фильма, сериала
- `rating` - возрастной рейтинг
- `ratingLevel` - детализация, описание возрастного рейтинга
- `ratingDescription` - числовая кодировка рейтинга
- `release year` - дата выхода фильма или сериала
- `user rating score` - оценка контента
- `user rating size` - популярность контента

Признаки `ratingLevel` и `ratingDescription` - являются дополнительными кодировками к `rating`. Для целей анализа планируем использовать только признак `rating`, поэтому удаляем их сразу.


In [78]:
print("Значения user rating score:", data["user rating score"].dropna().unique()[:20])
print(data["user rating score"].describe())
print("-----------------")
print("Значения user rating size:", data["user rating size"].dropna().unique()[:20])
print(data["user rating size"].describe())


Значения user rating score: [82. 98. 94. 95. 97. 91. 96. 77. 88. 80. 74. 81. 57. 84. 83. 99. 89. 92.
 62. 90.]
count    605.000000
mean      84.094215
std       12.344371
min       55.000000
25%       75.000000
50%       88.000000
75%       95.000000
max       99.000000
Name: user rating score, dtype: float64
-----------------
Значения user rating size: [80 82 81]
count    1000.000000
mean       80.783000
std         0.973066
min        80.000000
25%        80.000000
50%        80.000000
75%        82.000000
max        82.000000
Name: user rating size, dtype: float64


Также понять природу признаков `user rating score` и `user rating size` не представляется возможным, так как не известен источник этих оценок.

`user rating size` принимает только три значения: 80, 81, 82. Признак практически константа, у него дисперсия близка к нулю, информационной ценности нет.

Дополнительно стоит отметить, что на платформе Netflix не существует общепринятых оценок по бальной шкале, они используют индивидуальную систему мэтчей. Поэтому было принято решение также удалить эти признаки. В следующих частях планируем использовать рейтинг с платформы IMDB.

In [79]:
data = data.drop(
    columns=[
        "ratingLevel",
        "ratingDescription",
        "user rating score",
        "user rating size",
    ]
)
data.head(3)

,title,rating,release year
0,White Chicks,PG-13,2004
1,Lucky Number Slevin,R,2006
2,Grey's Anatomy,TV-14,2016


## Шаг 2. Дубликаты

При рассмотрении данных заметно, что одни и те же шоу встречаются несколько раз.
Часть из них - это сериалы. Также возможно данные были собраны парсингом или несколькими участниками, либо склейка файлов без проверки.


In [80]:
data[data["title"].str.lower() == "black mirror"]

,title,rating,release year
17,Black Mirror,TV-MA,2016
74,Black Mirror,TV-MA,2016
480,Black Mirror,TV-MA,2016


In [81]:
data[data.duplicated()].sort_values("title").head(20)

,title,rating,release year
396,13 Reasons Why,TV-MA,2017
241,13 Reasons Why,TV-MA,2017
347,13 Reasons Why,TV-MA,2017
141,13 Reasons Why,TV-MA,2017
295,13 Reasons Why,TV-MA,2017
497,13 Reasons Why,TV-MA,2017
189,13 Reasons Why,TV-MA,2017
201,30 Rock,TV-14,2012
218,5 to 7,R,2014
265,90210,TV-14,2013


Всего строк в файле 1000. Из них дублей - 500. Ровно половина датасета это дубликаты.


In [82]:
print(data.shape[0])
print(data.duplicated().sum())

1000
500


Применяем `drop_duplicates()`, потому что задублированы именно целые строки. После у нас остается 500 уникальных строк.

По каждому сериалу оставим одну строку, так как оги ничем не отличаются.


In [83]:
data_no_duplicates = data.drop_duplicates()
data_no_duplicates.shape[0]

500

Также необходимо сбросить индекс и переименовываем датафрейм, чтобы не путаться. Дальше работает с датасетом `data_clear`


In [84]:
data_no_duplicates.tail(5)

,title,rating,release year
989,Russell Madness,PG,2015
993,Wiener Dog Internationals,G,2015
994,Pup Star,G,2016
997,Precious Puppies,TV-G,2003
998,Beary Tales,TV-G,2013


In [85]:
data_clear = data_no_duplicates.copy().reset_index(drop=True)
data_clear.tail(5)

,title,rating,release year
495,Russell Madness,PG,2015
496,Wiener Dog Internationals,G,2015
497,Pup Star,G,2016
498,Precious Puppies,TV-G,2003
499,Beary Tales,TV-G,2013


## Шаг 3. Пропуски (NaN)


In [86]:
data_clear.isnull().sum()

title           0
rating          0
release year    0
dtype: int64

По оставшимся признакам пропусков нет.


## Шаг 4. Формат названий колонок (признаков) и значений

Приведем названия признаков к единому формату. Выбрали snake_case


In [87]:
data_clear = data_clear.rename(
    columns={
        "release year": "release_year",
    }
)
data_clear.columns.tolist()

['title', 'rating', 'release_year']

Для дальнейшей работы перед слиянием с другими датасетами дополнительно подготовим значения

- Приводим год выпуска `release_year` к числу для дальнейшего сравнения
- Заголовок `title` приводим к нижнему регистру для точного сравнения
- Убираем дубли по заголовку `title` и году выпуска `release_year`


In [88]:
data_clear["title_lower"] = data_clear["title"].str.lower().str.strip()
data_clear["release_year"] = data_clear["release_year"].astype("Int64")

data_clear = data_clear.drop_duplicates(subset=["title", "release_year"]).reset_index(
    drop=True
)

**Финальная проверка**


In [89]:
data_clear.isna().sum()

title           0
rating          0
release_year    0
title_lower     0
dtype: int64

## Шаг 5. Сохранение результата

Сохраняем итоговый датасет. Он будет использоваться далее для обогащения данными из внешних источников.


In [90]:
data_clear.to_csv("netflix_clean_1.csv", index=False)

## Итоги предобработки

Итоговый датасет: **499 строк, 4 признака**.


In [91]:
data_clear.shape

(499, 4)